In [16]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent.parent))  # add repo root
from rsr import rsr
import torch
import json

from ndtools import fun_binary_graph as fbg # ndtools available at github.com/jieunbyun/network-datasets
from ndtools.graphs import build_graph
from pathlib import Path
import networkx as nx   

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# Define Inputs

## System function

Load json files that are required to define the system function.
The files are loaded as dictionary.

In [17]:
DATASET = Path("data") 

nodes = json.loads((DATASET / "nodes.json").read_text(encoding="utf-8"))
edges = json.loads((DATASET / "edges.json").read_text(encoding="utf-8"))
probs_dict = json.loads((DATASET / "probs.json").read_text(encoding="utf-8"))

The system is a network.

<img src="data/graph.png" width="500">

Since the system is a network, we use ndtools to define the system function.

In [18]:
# build base graph
G_base: nx.Graph = build_graph(nodes, edges, probs_dict)

The system performance is measured by global connectivity. 
For instance, when all components have state 1 (i.e. in operation), the graph has global connectivity of 2.

In [19]:
# all edges ON (example); add node/edge 0s as needed
comps_st = {eid: 1 for eid in edges.keys()}
# System performance is measured by global connectivity.
k_val, state, _ = fbg.eval_global_conn_k(comps_st, G_base)

print("state: ", comps_st)
print("k =", k_val, "state =", state)

state:  {'e01': 1, 'e02': 1, 'e03': 1, 'e04': 1, 'e05': 1, 'e06': 1, 'e07': 1, 'e08': 1, 'e09': 1, 'e10': 1, 'e11': 1}
k = 2 state = 2


We use this function fbg.eval_global_conn_k to define s_fun.

There is a strict rule in defining s_fun.
* It must accept one input: comps_st, which is a dictionary as defined above.
* It must return three outputs.
    - fval: the function value (e.g. performance).
    - state: integer system state
    - ref_state: minimal reference state - if unavailable, return None.

In [20]:
# s_fun must accept comps_st and return (fval, state, <ref_state>),
# where fval is the function value (e.g. performance); state is the system's discrete state (e.g. performance level).
s_fun = lambda comps_st: fbg.eval_global_conn_k(comps_st, G_base)
row_names = list(edges.keys()) 
n_state = 2  # binary states: 0, 1

## Component probabilities

Cast the dictionary variable into a tensor for RSR implementation.

In [21]:
probs = [[probs_dict[n]['0']['p'], probs_dict[n]['1']['p']] for n in row_names]
probs = torch.tensor(probs, dtype=torch.float32, device=device)

# Obtain reference states

In [22]:
sys_upper_st = 2 # either 1 or 2

result = rsr.run_ref_extraction_by_mcs(
    # Problem-specific callables / data
    sfun=s_fun,
    probs=probs,
    row_names=row_names,
    n_state=n_state,
    sys_upper_st=sys_upper_st,
    unk_prob_thres = 1e-6,
    output_dir="rsr_res"

) 

---
Round: 1, Unk. prob.: 1.000e+00
Upper probs: 0.000e+00, Lower probs: 0.000e+00
No. of non-dominant refs: 0, Survival refs: 0, Failure refs: 0
Survival sample found from sampling.
No. of existing refs removed:  0
New ref added. System state: 2, System value: 2. Total samples: 100000.
New ref (No. of conditions: 8): {'e02': ('>=', 1), 'e03': ('>=', 1), 'e04': ('>=', 1), 'e05': ('>=', 1), 'e07': ('>=', 1), 'e08': ('>=', 1), 'e09': ('>=', 1), 'e11': ('>=', 1), 'sys': ('>=', 2)}
Updated sys_vals: [2]
---
Round: 2, Unk. prob.: 1.000e+00
Upper probs: 0.000e+00, Lower probs: 0.000e+00
No. of non-dominant refs: 1, Survival refs: 1, Failure refs: 0
Failure sample found from sampling.
No. of existing refs removed:  0
New ref added. System state: 0, System value: 1. Total samples: 100000.
New ref (No. of conditions: 1): {'e09': ('<=', 0), 'sys': ('<=', 1)}
Updated sys_vals: [1, 2]
---
Round: 3, Unk. prob.: 8.321e-01
Upper probs: 1.679e-01, Lower probs: 0.000e+00
No. of non-dominant refs: 2, Su